## Generate FANJIAN

In [ ]:
import re

pat1 = re.compile(r"<[ks].*?$")

with open('./Unihan_Variants.txt') as file:
    line_list = [pat1.sub('', line).split() for line in file if line.startswith("U+")]


pat2 = re.compile(r"(?P<unicode>[\d\.A-Z]+?)\s+;\s+(?P<value>[\dA-Z]+)\s+#")

with open("./EquivalentUnifiedIdeograph.txt") as file:
    for line in file:
        if line == "\n" or line.startswith("#"):
            continue
        d = pat2.search(line)
        if d is None:
            continue
        if ".." in d["unicode"]:
            start = d["unicode"].split("..")[0]
            end = d["unicode"].split("..")[1]
            line_list.extend(["U+" + hex(i)[2:].upper(), "kEquivalent", "U+" + d["value"]] for i in range(int(start, 16), int(end, 16) + 1))
        else:
            line_list.append(["U+" + d["unicode"], "kEquivalent", "U+" + d["value"]])
file.close()

In [39]:
order = {
    "kSimplifiedVariant": 1,
    "kTraditionalVariant": 2,
    "kEquivalent": 3,
    "kSemanticVariant": 4,
    "kSpecializedSemanticVariant": 5,
    "kZVariant": 6,
    "kSpoofingVariant": 7,
}

line_list.sort(key=lambda x: (order[x[1]], x[0]))

m = {}
for pair in line_list:
    if pair[1] == "kTraditionalVariant":
        pair[0], pair[2] = pair[2], pair[0]

    if m.get(pair[2], '') == pair[0] or m.get(pair[0]):
        continue

    elif m.get(pair[2]):
        m[pair[0]] = m[pair[2]]

    else:
        m[pair[0]] = pair[2]

m = {
    chr(int(k.replace("U+", ""), 16)): chr(int(v.replace("U+", ""), 16))
    for k, v in m.items()
    if k != v
}

with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in m.items()]))

In [40]:
mm = m

## Generate NUM-NORM

In [ ]:
import re

pat = re.compile(r"(?P<unicode>[\d\.A-Z]+?)\s+.*?; (?P<value>[\-\d\/]+) #")

with open("./DerivedNumericValues.txt") as file:
    m = {}
    for line in file:
        if line == "\n" or line.startswith("#"):
            continue
        d = pat.search(line)
        if d is None:
            continue
        if ".." in d["unicode"]:
            start = d["unicode"].split("..")[0]
            end = d["unicode"].split("..")[1]
            for i in range(int(start, 16), int(end, 16) + 1):
                m[chr(i)] = d["value"]
        else:
            m[chr(int(d["unicode"], 16))] = d["value"]
file.close()

with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in m.items()]))

## Generate TEXTDELETE

In [ ]:
import re
import unicodedata

pat = re.compile(r"^(?P<unicode>[\d\.A-Z]+?)\s*?; (?P<value>[A-Za-z]{2}) #")

with open("./DerivedGeneralCategory.txt") as file:
    d = []
    n = {}
    for line in file:
        if line == "\n" or line.startswith("#"):
            continue
        m = pat.search(line)
        if m is None:
            continue
        m = m.groupdict()
        if m["value"] in [
            "Mc",
            "Me",
            "Mn",
            "Pc",
            "Pd",
            "Pe",
            "Pf",
            "Pi",
            "Po",
            "Ps",
            "Sc",
            "Sk",
            "Sm",
            "So",
        ]:
            if ".." in m["unicode"]:
                start = m["unicode"].split("..")[0]
                end = m["unicode"].split("..")[1]
                for i in range(int(start, 16), int(end, 16) + 1):
                    x = unicodedata.decomposition(chr(i))
                    if x:
                        n[chr(i)] = "".join([
                            chr(int(y, 16))
                            for y in unicodedata.decomposition(chr(i)).split(" ")[1:]
                        ])
                    else:
                        d.append(chr(i))
            else:
                unicode = m["unicode"]
                x = unicodedata.decomposition(chr(int(unicode, 16)))
                if x:
                    n[chr(int(unicode, 16))] = "".join([
                        chr(int(y, 16))
                        for y in unicodedata.decomposition(chr(int(unicode, 16))).split(
                            " "
                        )[1:]
                    ])
                else:
                    d.append(chr(int(unicode, 16)))

with open("./testtest.txt", "w") as f:
    f.write("\n".join(d))

In [43]:
with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in n.items()]))

## Generate PINYIN

In [ ]:
import re

import unidecode

pat = re.compile(r"^U\+(?P<unicode>.+?)\s+kMandarin\s+(?P<value>.*?)$")

with open("./Unihan_Readings.txt") as file:
    pinyin = {}
    for line in file.read().splitlines():
        m = pat.search(line)
        if m:
            d = m.groupdict()
            pinyin[chr(int(d["unicode"], 16))] = unidecode.unidecode(d["value"])

with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t␀{v}␀" for k, v in pinyin.items()]))

## GENERATE EN VARIATION

In [ ]:
import re

import unidecode

pat = re.compile(r"^(?P<unicode>[0-9A-Z]+?)\s+; L[lu] #.*?$")

with open("./DerivedGeneralCategory.txt") as file:
    d = {}
    for line in file:
        m = pat.search(line)
        if m:
            unicode = m.groupdict()["unicode"]
            if ".." in unicode:
                start = unicode.split("..")[0]
                end = unicode.split("..")[1]
                for i in range(int(start, 16), int(end, 16) + 1):
                    d[chr(i)] = (
                        unidecode.unidecode(chr(i)).lower().replace("'", "")
                        if unidecode.unidecode(chr(i))
                        else chr(i).lower()
                    )
            else:
                d[chr(int(unicode, 16))] = (
                    unidecode.unidecode(chr(int(unicode, 16))).lower().replace("'", "")
                    if unidecode.unidecode(chr(int(unicode, 16)))
                    else chr(int(unicode, 16)).lower()
                )


In [49]:
with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in d.items() if k != v]))

In [ ]:
import re

import unidecode

with open("./NormalizationTest.txt") as file:
    d = {}
    for line in file:
        if " HANGUL " in line or line.startswith(("#", "@")):
            continue
        m = line.split(";")
        if len(m[0].split(" ")) > 1:
            continue
        d[chr(int(m[0], 16))] = "␀".join([
            chr(int(v, 16)).lower() for v in m[4].split(" ")
        ])

In [71]:
with open("./testtest.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in d.items() if k != v]))

In [ ]:
with open("../../matcher_rs/process_map/NORM.txt") as f:
    m = {}
    for line in f.read().splitlines():
        k, v = line.split()
        m[k] = v

with open("../../matcher_rs/process_map/NORM_.txt", "w") as f:
    f.write("\n".join([f"{k}\t{v}" for k, v in m.items()]))

## Test

In [ ]:
with open('../../matcher_rs/process_map/FANJIAN.txt') as fanjian_file:
    fanjian_map = {
        k: v for k, v in [line.split("\t") for line in fanjian_file.read().splitlines()]
    }

In [ ]:
with open("../../matcher_rs/process_map/FANJIAN_.txt", "w") as fanjian_file:
    fanjian_file.write("\n".join([f"{k}\t{v}" for k,v in fanjian_map.items()]))